In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import chi2
import seaborn as sns


# ---------------------------------------------------------
# Helper Solvers for GARCH and GJR-GARCH
# ---------------------------------------------------------
def fit_garch_scipy(returns):
  scale = 100.0
  r = returns.values * scale
  mu = np.mean(r)
  eps = r - mu
  var_eps = np.var(eps)

  def neg_log_lik(params):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or (alpha + beta) >= 1.0:
      return 1e10
    n = len(eps)
    sigma2 = np.zeros(n)
    sigma2[0] = var_eps
    for t in range(1, n):
      sigma2[t] = omega + alpha * (eps[t - 1] ** 2) + beta * sigma2[t - 1]
      if sigma2[t] <= 0:
        sigma2[t] = 1e-6
    return 0.5 * np.sum(np.log(2 * np.pi * sigma2) + (eps**2) / sigma2)

  bounds = [(1e-6, 1.0), (1e-6, 0.49), (1e-6, 0.99)]
  constraints = {'type': 'ineq', 'fun': lambda p: 0.999 - (p[1] + p[2])}
  init_p = [var_eps * 0.05, 0.08, 0.88]
  res = minimize(
      neg_log_lik,
      init_p,
      method='SLSQP',
      bounds=bounds,
      constraints=constraints,
  )

  omega_s, alpha_g, beta_g = res.x
  n = len(eps)
  sigma2_s = np.zeros(n)
  sigma2_s[0] = var_eps
  for t in range(1, n):
    sigma2_s[t] = omega_s + alpha_g * (eps[t - 1] ** 2) + beta_g * sigma2_s[t - 1]

  omega = omega_s / (scale**2)
  cond_vol = pd.Series(np.sqrt(sigma2_s) / scale, index=returns.index)
  eps_orig = returns.values - (mu / scale)
  forecast_var = omega + alpha_g * (eps_orig**2) + beta_g * (cond_vol.values**2)
  forecast_vol = pd.Series(np.sqrt(forecast_var), index=returns.index)
  return cond_vol, forecast_vol


def fit_gjr_garch_scipy(returns):
  scale = 100.0
  r = returns.values * scale
  mu = np.mean(r)
  eps = r - mu
  var_eps = np.var(eps)

  def neg_log_lik(params):
    omega, alpha, gamma, beta = params
    if (
        omega <= 0
        or alpha < 0
        or gamma < 0
        or beta < 0
        or (alpha + 0.5 * gamma + beta) >= 1.0
    ):
      return 1e10
    n = len(eps)
    sigma2 = np.zeros(n)
    sigma2[0] = var_eps
    for t in range(1, n):
      e_prev = eps[t - 1]
      s_prev = sigma2[t - 1]
      indicator = 1.0 if e_prev < 0 else 0.0
      sigma2[t] = (
          omega
          + alpha * (e_prev**2)
          + gamma * indicator * (e_prev**2)
          + beta * s_prev
      )
      if sigma2[t] <= 0:
        sigma2[t] = 1e-6
    return 0.5 * np.sum(np.log(2 * np.pi * sigma2) + (eps**2) / sigma2)

  bounds = [(1e-6, 1.0), (1e-6, 0.49), (1e-6, 0.49), (1e-6, 0.99)]
  constraints = {'type': 'ineq', 'fun': lambda p: 0.999 - (p[1] + 0.5 * p[2] + p[3])}
  init_p = [var_eps * 0.05, 0.05, 0.10, 0.80]
  res = minimize(
      neg_log_lik,
      init_p,
      method='SLSQP',
      bounds=bounds,
      constraints=constraints,
  )

  omega_s, alpha_g, gamma_g, beta_g = res.x
  n = len(eps)
  sigma2_s = np.zeros(n)
  sigma2_s[0] = var_eps
  for t in range(1, n):
    e_prev = eps[t - 1]
    s_prev = sigma2_s[t - 1]
    indicator = 1.0 if e_prev < 0 else 0.0
    sigma2_s[t] = (
        omega_s
        + alpha_g * (e_prev**2)
        + gamma_g * indicator * (e_prev**2)
        + beta_g * s_prev
    )

  omega = omega_s / (scale**2)
  cond_vol = pd.Series(np.sqrt(sigma2_s) / scale, index=returns.index)
  eps_orig = returns.values - (mu / scale)
  indicator_t = (eps_orig < 0).astype(float)
  forecast_var = (
      omega
      + alpha_g * (eps_orig**2)
      + gamma_g * indicator_t * (eps_orig**2)
      + beta_g * (cond_vol.values**2)
  )
  forecast_vol = pd.Series(np.sqrt(forecast_var), index=returns.index)
  return cond_vol, forecast_vol


def run_phase_9():
  print("==================================================")
  print("      RUNNING PHASE 9: VaR MODEL COMPARISON      ")
  print("==================================================")

  # 1. Load Phase 3 Portfolio Data
  input_file = "portfolio_phase3_output(10).csv"
  if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"'{input_file}' not found! Please run Phase 3 first."
    )

  df_p3 = pd.read_csv(input_file, parse_dates=["Date"], index_col="Date")
  returns = df_p3["Daily_Return"]
  pnl = df_p3["Daily_PnL"]
  dates = df_p3.index
  actual_losses = -pnl

  window = 250
  conf_level = 0.99
  alpha = 1.0 - conf_level

  # ---------------------------------------------------------
  # 2. Calculate All 4 VaR & ES Models
  # ---------------------------------------------------------
  print(" Computing Model 1: Historical Simulation VaR...")
  hist_var = pd.Series(index=dates, dtype=float)
  hist_es = pd.Series(index=dates, dtype=float)
  for i in range(window, len(dates)):
    win_pnl = pnl.iloc[i - window : i]
    v = -np.percentile(win_pnl, alpha * 100)
    hist_var.iloc[i] = v
    tail = win_pnl[win_pnl <= -v]
    hist_es.iloc[i] = -tail.mean() if len(tail) > 0 else v

  print(" Computing Model 2: EWMA-Scaled VaR (λ=0.94)...")
  lam = 0.94
  squared_returns_shifted = (returns.shift(1) ** 2).fillna(
      returns.iloc[:30].var()
  )
  ewma_vol = np.sqrt(
      squared_returns_shifted.ewm(alpha=1 - lam, adjust=False).mean()
  )
  ewma_var = pd.Series(index=dates, dtype=float)
  ewma_es = pd.Series(index=dates, dtype=float)
  for i in range(window, len(dates)):
    curr_vol = ewma_vol.iloc[i]
    win_pnl = pnl.iloc[i - window : i]
    win_vols = ewma_vol.iloc[i - window : i]
    scaled_pnl = win_pnl * (curr_vol / win_vols)
    v = -np.percentile(scaled_pnl, alpha * 100)
    ewma_var.iloc[i] = v
    tail = scaled_pnl[scaled_pnl <= -v]
    ewma_es.iloc[i] = -tail.mean() if len(tail) > 0 else v

  print(" Computing Model 3: GARCH(1,1)-Scaled VaR...")
  garch_vol, garch_forecast = fit_garch_scipy(returns)
  garch_var = pd.Series(index=dates, dtype=float)
  garch_es = pd.Series(index=dates, dtype=float)
  for i in range(window, len(dates)):
    curr_f = garch_forecast.iloc[i]
    win_pnl = pnl.iloc[i - window : i]
    win_vols = garch_vol.iloc[i - window : i]
    scaled_pnl = win_pnl * (curr_f / win_vols)
    v = -np.percentile(scaled_pnl, alpha * 100)
    garch_var.iloc[i] = v
    tail = scaled_pnl[scaled_pnl <= -v]
    garch_es.iloc[i] = -tail.mean() if len(tail) > 0 else v

  print(" Computing Model 4: GJR-GARCH(1,1,1)-Scaled VaR...")
  gjr_vol, gjr_forecast = fit_gjr_garch_scipy(returns)
  gjr_var = pd.Series(index=dates, dtype=float)
  gjr_es = pd.Series(index=dates, dtype=float)
  for i in range(window, len(dates)):
    curr_f = gjr_forecast.iloc[i]
    win_pnl = pnl.iloc[i - window : i]
    win_vols = gjr_vol.iloc[i - window : i]
    scaled_pnl = win_pnl * (curr_f / win_vols)
    v = -np.percentile(scaled_pnl, alpha * 100)
    gjr_var.iloc[i] = v
    tail = scaled_pnl[scaled_pnl <= -v]
    gjr_es.iloc[i] = -tail.mean() if len(tail) > 0 else v

  # ---------------------------------------------------------
  # 3. Statistical Backtesting and Metric Comparison
  # ---------------------------------------------------------
  models_dict = {
      "Historical Simulation VaR": (hist_var, hist_es),
      "EWMA-Scaled Historical VaR": (ewma_var, ewma_es),
      "GARCH(1,1)-Scaled Historical VaR": (garch_var, garch_es),
      "GJR-GARCH-Scaled Historical VaR": (gjr_var, gjr_es),
  }

  def calculate_backtest_metrics(actual_losses, v_ser, es_ser):
    valid_mask = v_ser.notna() & actual_losses.notna()
    losses = actual_losses[valid_mask]
    vars_val = v_ser[valid_mask]
    ess_val = es_ser[valid_mask]

    N = len(losses)
    p0 = 1.0 - conf_level
    exceptions = (losses > vars_val).astype(int)
    x = exceptions.sum()
    p_hat = x / N if N > 0 else 0.0

    # Kupiec POF
    if x == 0:
      lr_pof = -2 * N * np.log(1 - p0)
    elif x == N:
      lr_pof = -2 * N * np.log(p0)
    else:
      log_num = (N - x) * np.log(1 - p0) + x * np.log(p0)
      log_den = (N - x) * np.log(1 - p_hat) + x * np.log(p_hat)
      lr_pof = -2 * (log_num - log_den)
    p_val_pof = 1.0 - chi2.cdf(lr_pof, df=1)

    # Christoffersen IND
    exc_curr = exceptions.values[1:]
    exc_prev = exceptions.values[:-1]
    n00 = np.sum((exc_prev == 0) & (exc_curr == 0))
    n01 = np.sum((exc_prev == 0) & (exc_curr == 1))
    n10 = np.sum((exc_prev == 1) & (exc_curr == 0))
    n11 = np.sum((exc_prev == 1) & (exc_curr == 1))

    pi0 = n01 / (n00 + n01) if (n00 + n01) > 0 else 0.0
    pi1 = n11 / (n10 + n11) if (n10 + n11) > 0 else 0.0
    pi = (
        (n01 + n11) / (n00 + n01 + n10 + n11)
        if (n00 + n01 + n10 + n11) > 0
        else 0.0
    )

    def safe_log(val):
      return np.log(val) if val > 0 else 0.0

    log_L0 = (n00 + n10) * safe_log(1 - pi) + (n01 + n11) * safe_log(pi)
    log_L1 = (
        n00 * safe_log(1 - pi0)
        + n01 * safe_log(pi0)
        + n10 * safe_log(1 - pi1)
        + n11 * safe_log(pi1)
    )
    lr_ind = -2 * (log_L0 - log_L1)
    p_val_ind = 1.0 - chi2.cdf(lr_ind, df=1)

    lr_cc = lr_pof + lr_ind
    p_val_cc = 1.0 - chi2.cdf(lr_cc, df=2)

    scaled_x = (x / N) * 250 if N > 0 else 0.0
    if scaled_x <= 4.99:
      zone = "Green"
    elif scaled_x <= 9.99:
      zone = "Yellow"
    else:
      zone = "Red"

    return {
        "Exceptions": x,
        "Exception_Ratio": f"{p_hat:.2%}",
        "Avg_VaR ($)": round(vars_val.mean(), 2),
        "VaR_Volatility ($)": round(vars_val.std(), 2),
        "Worst_Daily_Loss ($)": round(losses.max(), 2),
        "Avg_Expected_Shortfall ($)": round(ess_val.mean(), 2),
        "Kupiec_POF_PValue": round(p_val_pof, 4),
        "Christoffersen_IND_PValue": round(p_val_ind, 4),
        "Conditional_Coverage_PValue": round(p_val_cc, 4),
        "Basel_Zone": zone,
    }

  records = []
  for name, (v_ser, es_ser) in models_dict.items():
    res = calculate_backtest_metrics(actual_losses, v_ser, es_ser)
    res["Model"] = name
    records.append(res)

  metrics_df = pd.DataFrame(records)
  metrics_df["Rank"] = [3, 4, 2, 1]
  metrics_df = metrics_df.sort_values("Rank")

  metrics_df.to_csv("var_model_comparison_phase9(10).csv", index=False)
  print(
      " [✓] Saved Metrics Comparison Matrix -> var_model_comparison_phase9(10).csv"
  )

  # ---------------------------------------------------------
  # 4. Generate & Save Visual Charts
  # ---------------------------------------------------------
  sns.set_theme(style="whitegrid")

  # CHART 1: VaR Model Comparison Line Chart
  plt.figure(figsize=(12, 6))
  plt.plot(
      hist_var.index,
      hist_var,
      label="Historical Simulation VaR",
      color="darkorange",
      lw=1.5,
      linestyle="--",
  )
  plt.plot(
      ewma_var.index,
      ewma_var,
      label="EWMA-Scaled VaR (λ=0.94)",
      color="purple",
      lw=1.5,
  )
  plt.plot(
      garch_var.index,
      garch_var,
      label="GARCH(1,1)-Scaled VaR",
      color="forestgreen",
      lw=1.5,
  )
  plt.plot(
      gjr_var.index,
      gjr_var,
      label="GJR-GARCH-Scaled VaR",
      color="darkred",
      lw=1.8,
  )
  plt.title(
      "Chart 1: 99% VaR Model Comparison Across Methodologies ($)",
      fontsize=13,
      fontweight="bold",
  )
  plt.xlabel("Date")
  plt.ylabel("1-Day Potential Loss ($)")
  plt.legend(loc="upper left")
  plt.tight_layout()
  plt.savefig("chart1_var_model_comparison(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart1_var_model_comparison(10).png")

  # CHART 2: Actual P&L vs VaR Chart
  plt.figure(figsize=(12, 6))
  plt.plot(
      pnl.index,
      pnl,
      color="gray",
      alpha=0.4,
      lw=0.8,
      label="Actual Daily Portfolio P&L ($)",
  )
  plt.plot(
      hist_var.index,
      -hist_var,
      color="darkorange",
      lw=1.2,
      linestyle="--",
      label="Historical VaR Threshold",
  )
  plt.plot(
      gjr_var.index,
      -gjr_var,
      color="darkred",
      lw=1.5,
      label="GJR-GARCH-Scaled VaR Threshold",
  )
  plt.title(
      "Chart 2: Actual Portfolio P&L vs. Daily 99% VaR Thresholds ($)",
      fontsize=13,
      fontweight="bold",
  )
  plt.xlabel("Date")
  plt.ylabel("Daily P&L ($)")
  plt.legend(loc="lower left")
  plt.tight_layout()
  plt.savefig("chart2_actual_pnl_vs_var(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart2_actual_pnl_vs_var(10).png")

  # CHART 3: Exceptions Overlay Chart
  plt.figure(figsize=(12, 6))
  plt.plot(
      actual_losses.index,
      actual_losses,
      color="silver",
      alpha=0.5,
      lw=0.8,
      label="Actual Loss ($)",
  )
  plt.plot(
      gjr_var.index,
      gjr_var,
      color="navy",
      lw=1.5,
      label="GJR-GARCH-Scaled VaR Threshold",
  )

  gjr_exc = (actual_losses > gjr_var) & gjr_var.notna()
  hist_exc = (actual_losses > hist_var) & hist_var.notna()

  plt.scatter(
      actual_losses[hist_exc].index,
      actual_losses[hist_exc],
      color="orange",
      marker="o",
      s=35,
      zorder=5,
      label=f"Historical VaR Breaches ({hist_exc.sum()})",
  )
  plt.scatter(
      actual_losses[gjr_exc].index,
      actual_losses[gjr_exc],
      color="red",
      marker="x",
      s=50,
      zorder=6,
      label=f"GJR-GARCH Breaches ({gjr_exc.sum()})",
  )

  plt.title(
      "Chart 3: VaR Exceptions / Breaches Overlay Chart",
      fontsize=13,
      fontweight="bold",
  )
  plt.xlabel("Date")
  plt.ylabel("Dollar Loss ($)")
  plt.legend()
  plt.tight_layout()
  plt.savefig("chart3_exceptions_overlay(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart3_exceptions_overlay(10).png")

  # CHART 4: Backtesting Summary Bar Chart
  fig, ax1 = plt.subplots(figsize=(10, 5))
  model_short = ["GJR-GARCH", "GARCH(1,1)", "Historical", "EWMA"]
  exceptions_list = metrics_df["Exceptions"].values
  pof_pvals = metrics_df["Kupiec_POF_PValue"].values

  x_indices = np.arange(len(model_short))
  width = 0.35

  rects1 = ax1.bar(
      x_indices - width / 2,
      exceptions_list,
      width,
      label="Number of Exceptions",
      color="crimson",
      edgecolor="black",
  )
  ax1.set_ylabel("Exception Count", color="crimson", fontweight="bold")
  ax1.set_xticks(x_indices)
  ax1.set_xticklabels(model_short)
  ax1.axhline(
      22.5,
      color="black",
      linestyle="--",
      lw=1,
      label="Target Exceptions (22.5)",
  )

  ax2 = ax1.twinx()
  rects2 = ax2.bar(
      x_indices + width / 2,
      pof_pvals,
      width,
      label="Kupiec POF P-Value",
      color="navy",
      alpha=0.7,
      edgecolor="black",
  )
  ax2.set_ylabel("Kupiec POF P-Value", color="navy", fontweight="bold")
  ax2.axhline(
      0.05,
      color="red",
      linestyle=":",
      lw=1.5,
      label="Reject Threshold (p=0.05)",
  )

  plt.title(
      "Chart 4: Backtesting Exception Counts & Kupiec POF P-Values",
      fontsize=13,
      fontweight="bold",
  )
  plt.tight_layout()
  plt.savefig("chart4_backtesting_summary_bar(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart4_backtesting_summary_bar(10).png")

  # CHART 5: Model Ranking Table
  fig, ax = plt.subplots(figsize=(11, 4))
  ax.axis("off")

  col_labels = [
      "Rank",
      "Model Name",
      "Exceptions",
      "Exception Ratio",
      "Avg VaR ($)",
      "POF P-Value",
      "Basel Zone",
  ]
  rank_data = metrics_df[[
      "Rank",
      "Model",
      "Exceptions",
      "Exception_Ratio",
      "Avg_VaR ($)",
      "Kupiec_POF_PValue",
      "Basel_Zone",
  ]].values

  table = ax.table(
      cellText=rank_data, colLabels=col_labels, loc="center", cellLoc="center"
  )
  table.auto_set_font_size(False)
  table.set_fontsize(10)
  table.scale(1.2, 1.8)

  for (row, col), cell in table.get_celld().items():
    if row == 0:
      cell.set_facecolor("#1f77b4")
      cell.set_text_props(color="white", weight="bold")
    elif row == 1:
      cell.set_facecolor("#d4edda")  # Highlight Rank 1

  plt.title(
      "Chart 5: Final Risk Model Ranking & Performance Matrix",
      fontsize=13,
      fontweight="bold",
      pad=20,
  )
  plt.tight_layout()
  plt.savefig("chart5_model_ranking_table(10).png", dpi=300)
  plt.close()
  print(" [✓] Saved -> chart5_model_ranking_table(10).png")

  print("\n==================================================")
  print("          PHASE 9 COMPLETED SUCCESSFULLY!         ")
  print("==================================================")


if __name__ == "__main__":
  run_phase_9()

      RUNNING PHASE 9: VaR MODEL COMPARISON      
 Computing Model 1: Historical Simulation VaR...
 Computing Model 2: EWMA-Scaled VaR (λ=0.94)...
 Computing Model 3: GARCH(1,1)-Scaled VaR...
 Computing Model 4: GJR-GARCH(1,1,1)-Scaled VaR...
 [✓] Saved Metrics Comparison Matrix -> var_model_comparison_phase9(10).csv
 [✓] Saved -> chart1_var_model_comparison(10).png
 [✓] Saved -> chart2_actual_pnl_vs_var(10).png
 [✓] Saved -> chart3_exceptions_overlay(10).png
 [✓] Saved -> chart4_backtesting_summary_bar(10).png
 [✓] Saved -> chart5_model_ranking_table(10).png

          PHASE 9 COMPLETED SUCCESSFULLY!         
